In [170]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
import os

In [171]:
tokenizer = tiktoken.get_encoding("gpt2")

In [172]:
with open("the_verdict.txt") as f:
    text = f.read()

tokens = tokenizer.encode(text)

In [173]:
len(tokens)

5776

In [174]:
context_length = 6
stride = 1

In [195]:
samples = tokens[:20]
for i in range(0, len(samples) - context_length, 1):
    print(samples[i:i+context_length], samples[i+1:i+context_length+1])

[3336, 33310, 35, 18379, 198, 198] [33310, 35, 18379, 198, 198, 15749]
[33310, 35, 18379, 198, 198, 15749] [35, 18379, 198, 198, 15749, 40417]
[35, 18379, 198, 198, 15749, 40417] [18379, 198, 198, 15749, 40417, 198]
[18379, 198, 198, 15749, 40417, 198] [198, 198, 15749, 40417, 198, 198]
[198, 198, 15749, 40417, 198, 198] [198, 15749, 40417, 198, 198, 40]
[198, 15749, 40417, 198, 198, 40] [15749, 40417, 198, 198, 40, 550]
[15749, 40417, 198, 198, 40, 550] [40417, 198, 198, 40, 550, 1464]
[40417, 198, 198, 40, 550, 1464] [198, 198, 40, 550, 1464, 1807]
[198, 198, 40, 550, 1464, 1807] [198, 40, 550, 1464, 1807, 3619]
[198, 40, 550, 1464, 1807, 3619] [40, 550, 1464, 1807, 3619, 402]
[40, 550, 1464, 1807, 3619, 402] [550, 1464, 1807, 3619, 402, 271]
[550, 1464, 1807, 3619, 402, 271] [1464, 1807, 3619, 402, 271, 10899]
[1464, 1807, 3619, 402, 271, 10899] [1807, 3619, 402, 271, 10899, 2138]
[1807, 3619, 402, 271, 10899, 2138] [3619, 402, 271, 10899, 2138, 257]


In [234]:
class GPTDataSetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        encodings = tokenizer.encode(text)

        for i in range(0, len(encodings) - context_length, stride):
            inputs = encodings[i:i+context_length]
            targets = encodings[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(inputs))
            self.target_ids.append(torch.tensor(targets))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [276]:
dataset = GPTDataSetV1(text[:200], tokenizer, 6, 1)

In [277]:
len(dataset)

49

In [315]:
def create_data_loader(text, batch_size=8, context_length=6, stride=3, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataSetV1(text, tokenizer, context_length, stride)
    n_cpus = os.cpu_count()
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=n_cpus)

In [316]:
dataloader = create_data_loader(text, batch_size=4, context_length=6, stride=1, shuffle=False)

In [317]:
len(dataloader)

1442

In [318]:
batch_iter = iter(dataloader)

In [319]:
a_batch = next(batch_iter)

In [324]:
inputs, targets =a_batch

In [334]:
for i in range(0, len(inputs)):
    print(inputs[i], targets[i])

tensor([ 3336, 33310,    35, 18379,   198,   198]) tensor([33310,    35, 18379,   198,   198, 15749])
tensor([33310,    35, 18379,   198,   198, 15749]) tensor([   35, 18379,   198,   198, 15749, 40417])
tensor([   35, 18379,   198,   198, 15749, 40417]) tensor([18379,   198,   198, 15749, 40417,   198])
tensor([18379,   198,   198, 15749, 40417,   198]) tensor([  198,   198, 15749, 40417,   198,   198])


In [335]:
dataset[:4]

([tensor([ 3336, 33310,    35, 18379,   198,   198]),
  tensor([33310,    35, 18379,   198,   198, 15749]),
  tensor([   35, 18379,   198,   198, 15749, 40417]),
  tensor([18379,   198,   198, 15749, 40417,   198])],
 [tensor([33310,    35, 18379,   198,   198, 15749]),
  tensor([   35, 18379,   198,   198, 15749, 40417]),
  tensor([18379,   198,   198, 15749, 40417,   198]),
  tensor([  198,   198, 15749, 40417,   198,   198])])